# 37. COMT 도킹 검증 (오버나이트)

## 목적
catechol 규칙의 rationale("COMT가 카테콜을 메틸화")을 실제 도킹으로
검증. 도파민(치환 전)과 메톡시 치환 후 분자를 COMT 구조(PDB 1VID)에
도킹해서 결합 스코어 변화를 확인.

## 배경 (37까지)
- 라이브러리 35개 규칙, 커버리지 33%+, 4-endpoint 검증, XGBoost 비교
  (RF가 4개 endpoint 전부 우세), 순서의존성 최종확정(85%/15%, 규칙기반
  3승0패)
- 규칙별 QED/LogP 분해: catechol 최대개선(+0.181), azo_A(324)만 유일하게
  QED 악화(-0.078) - 하이드라진 잔여반응성과 연결되는 발견
- 오늘 시연 이미지(case1~4) 확보, 아키텍처/4-endpoint/배치분포/커버리지
  성장곡선 차트 완성
- 스트레치 목표로 COMT 도킹 시도 (시간 오래 걸리면 오버나이트로 진행)
- test set은 여전히 미사용

## 다음에 할 것
- 도킹 결과 확인 후 제안서 반영 여부 판단
- 남은 커버리지 정리(바르비투레이트 부산물, ETRETINATE 등, 시간되면)

In [2]:
# 셀 1
!pip install rdkit -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q

In [3]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

fatal: destination path 'laidd-2026' already exists and is not an empty directory.
/content/laidd-2026
/content/laidd-2026


In [4]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [5]:
import importlib
from rdkit import Chem

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector

from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix

print(f"라이브러리 규칙 수: {len(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'])}")

라이브러리 규칙 수: 36


In [7]:
!pip install vina meeko -q
!wget -q https://files.rcsb.org/download/1VID.pdb -O receptor_raw.pdb
!ls -la receptor_raw.pdb

-rw-r--r-- 1 root root 184923 Aug  3 01:50 receptor_raw.pdb


In [8]:
with open("receptor_raw.pdb") as f:
    lines = f.readlines()

protein_lines = [l for l in lines if l.startswith(("ATOM", "TER", "END"))]
with open("receptor_protein.pdb", "w") as f:
    f.writelines(protein_lines)

hetero_lines = [l for l in lines if l.startswith("HETATM")]
print("HETATM(리간드/보조인자) 종류:", set(l[17:20].strip() for l in hetero_lines))

HETATM(리간드/보조인자) 종류: {'DNC', 'MG', 'SAM', 'HOH'}


In [9]:
with open(".gitignore", "a") as f:
    f.write("\n*.pdb\nreceptor_*\n")

print("완료")
!git status

완료
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	dopamine.pdbqt
	methoxy_dopamine.pdbqt
	receptor.pdbqt

no changes added to commit (use "git add" and/or "git commit -a")


In [10]:
dnc_lines = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == "DNC"]
print(f"DNC 원자 수: {len(dnc_lines)}")

coords = []
for l in dnc_lines:
    x, y, z = float(l[30:38]), float(l[38:46]), float(l[46:54])
    coords.append((x, y, z))

center_x = sum(c[0] for c in coords) / len(coords)
center_y = sum(c[1] for c in coords) / len(coords)
center_z = sum(c[2] for c in coords) / len(coords)

print(f"결합주머니 중심 좌표: ({center_x:.2f}, {center_y:.2f}, {center_z:.2f})")

DNC 원자 수: 14
결합주머니 중심 좌표: (-24.81, 60.00, 50.57)


In [11]:
!apt-get install -y openbabel -q 2>&1 | tail -3
!pip install gemmi -q
!obabel receptor_clean.pdb -O receptor.pdbqt -xr 2>&1 | tail -5
!ls -la receptor.pdbqt

Reading state information...
openbabel is already the newest version (3.1.1+dfsg-6ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is receptor_clean.pdb)

1 molecule converted
-rw-r--r-- 1 root root 134443 Aug  3 01:51 receptor.pdbqt


In [12]:
from meeko import MoleculePreparation
import subprocess

# 물 분자 제거하고 단백질+MG만 남긴 PDB 재생성
clean_lines = [l for l in lines if l.startswith(("ATOM", "TER", "END")) or (l.startswith("HETATM") and l[17:20].strip() == "MG")]
with open("receptor_clean.pdb", "w") as f:
    f.writelines(clean_lines)

!obabel receptor_clean.pdb -O receptor.pdbqt -xr 2>&1 | tail -5
!ls -la receptor.pdbqt

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is receptor_clean.pdb)

1 molecule converted


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-rw-r--r-- 1 root root 134443 Aug  3 01:51 receptor.pdbqt


In [13]:
from rdkit.Chem import AllChem

def prepare_ligand_pdbqt(smiles, filename):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    Chem.MolToPDBFile(mol, f"{filename}.pdb")
    subprocess.run(["obabel", f"{filename}.pdb", "-O", f"{filename}.pdbqt"], capture_output=True)
    return f"{filename}.pdbqt"

dopamine_smiles = "NCCc1ccc(O)c(O)c1"
methoxy_dopamine_smiles = propose_fix(dopamine_smiles, "catechol", candidate_idx=0)['new_smiles']
print("치환 후:", methoxy_dopamine_smiles)

lig1 = prepare_ligand_pdbqt(dopamine_smiles, "dopamine")
lig2 = prepare_ligand_pdbqt(methoxy_dopamine_smiles, "methoxy_dopamine")
print("준비 완료:", lig1, lig2)

치환 후: COc1ccc(CCN)cc1O
준비 완료: dopamine.pdbqt methoxy_dopamine.pdbqt


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
!apt-get install -y openbabel -q 2>&1 | tail -3
!pip install gemmi -q

!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina_bin
!chmod +x vina_bin
!./vina_bin --version

               total        used        free      shared  buff/cache   available
Mem:            12Gi       752Mi       7.5Gi       4.0Mi       4.5Gi        11Gi
Swap:             0B          0B          0B


In [16]:
!wget -q https://files.rcsb.org/download/1VID.pdb -O receptor_raw.pdb

with open("receptor_raw.pdb") as f:
    lines = f.readlines()

# 단백질 + MG(촉매 필수 금속)만 남기고 물 제거
clean_lines = [l for l in lines if l.startswith(("ATOM", "TER", "END"))
               or (l.startswith("HETATM") and l[17:20].strip() == "MG")]
with open("receptor_clean.pdb", "w") as f:
    f.writelines(clean_lines)

# 원래 결합된 리간드(DNC)로 결합주머니 중심 좌표 계산
dnc_lines = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == "DNC"]
coords = [(float(l[30:38]), float(l[38:46]), float(l[46:54])) for l in dnc_lines]
box_center = [sum(c[i] for c in coords) / len(coords) for i in range(3)]
box_size = [20, 20, 20]
print(f"결합주머니 중심 좌표: {box_center}")

!obabel receptor_clean.pdb -O receptor.pdbqt -xr 2>&1 | tail -3
!ls -la receptor.pdbqt

도파민: -5.72 kcal/mol


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess

def prepare_ligand_pdbqt(smiles, filename):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    Chem.MolToPDBFile(mol, f"{filename}.pdb")
    subprocess.run(["obabel", f"{filename}.pdb", "-O", f"{filename}.pdbqt"], capture_output=True)
    return f"{filename}.pdbqt"

dopamine_smiles = "NCCc1ccc(O)c(O)c1"
methoxy_dopamine_smiles = propose_fix(dopamine_smiles, "catechol", candidate_idx=0)['new_smiles']
print("치환 후:", methoxy_dopamine_smiles)

lig_dopamine = prepare_ligand_pdbqt(dopamine_smiles, "dopamine")
lig_methoxy = prepare_ligand_pdbqt(methoxy_dopamine_smiles, "methoxy_dopamine")
print("준비 완료:", lig_dopamine, lig_methoxy)

In [6]:
import re

def run_docking_cli(ligand_pdbqt, out_prefix, box_center, box_size, exhaustiveness=4):
    cmd = (f"./vina_bin --receptor receptor.pdbqt --ligand {ligand_pdbqt} "
           f"--center_x {box_center[0]} --center_y {box_center[1]} --center_z {box_center[2]} "
           f"--size_x {box_size[0]} --size_y {box_size[1]} --size_z {box_size[2]} "
           f"--exhaustiveness {exhaustiveness} --out {out_prefix}_out.pdbqt "
           f"> {out_prefix}_log.txt")
    result = os.system(cmd)
    with open(f"{out_prefix}_log.txt") as f:
        log = f.read()
    match = re.search(r"^\s*1\s+(-?\d+\.\d+)", log, re.MULTILINE)
    score = float(match.group(1)) if match else None
    return score, log

import os

score_dopamine, log_dopamine = run_docking_cli("dopamine.pdbqt", "dopamine", box_center, box_size)
print(f"도파민(치환전): {score_dopamine:.2f} kcal/mol")

score_methoxy, log_methoxy = run_docking_cli("methoxy_dopamine.pdbqt", "methoxy_dopamine", box_center, box_size)
print(f"메톡시-도파민(치환후): {score_methoxy:.2f} kcal/mol")

print(f"\n결합 스코어 변화: {score_methoxy - score_dopamine:+.2f} kcal/mol")
print("(더 음수=강한 결합, 양의 변화=결합력 약화)")

rm 'fpscores.pkl.gz'
rm 'sascorer.py'
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   .gitignore
	deleted:    fpscores.pkl.gz
	renamed:    batch_coordination_results.json -> outputs/batch_coordination_results.json
	renamed:    methoxy_dopamine_log.txt -> outputs/methoxy_dopamine_log.txt
	renamed:    order_comparison_progress.json -> outputs/order_comparison_progress.json
	renamed:    rule_level_qed_logp_breakdown.json -> outputs/rule_level_qed_logp_breakdown.json
	deleted:    sascorer.py



In [ ]:
!git add docs/experiment_results_log.md
!git status

In [7]:
!git commit -m "Reorganize repo root: move experiment result files (batch_coordination_results.json, methoxy_dopamine_log.txt, order_comparison_progress.json, rule_level_qed_logp_breakdown.json) into outputs/, remove accidentally-committed third-party downloads (sascorer.py, fpscores.pkl.gz from RDKit contrib) from tracking, add to .gitignore since these are re-downloaded via urlretrieve each session."
!git push origin main

[main 9b18a20] Reorganize repo root: move experiment result files (batch_coordination_results.json, methoxy_dopamine_log.txt, order_comparison_progress.json, rule_level_qed_logp_breakdown.json) into outputs/, remove accidentally-committed third-party downloads (sascorer.py, fpscores.pkl.gz from RDKit contrib) from tracking, add to .gitignore since these are re-downloaded via urlretrieve each session.
 7 files changed, 3 insertions(+), 192 deletions(-)
 delete mode 100644 fpscores.pkl.gz
 rename batch_coordination_results.json => outputs/batch_coordination_results.json (100%)
 rename methoxy_dopamine_log.txt => outputs/methoxy_dopamine_log.txt (100%)
 rename order_comparison_progress.json => outputs/order_comparison_progress.json (100%)
 rename rule_level_qed_logp_breakdown.json => outputs/rule_level_qed_logp_breakdown.json (100%)
 delete mode 100644 sascorer.py
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: